In [ ]:
# Import neccessary libraries
import warnings
warnings.filterwarnings("ignore")
import math, os, time, json
import numpy as np
import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup
from io import StringIO
from polygon import RESTClient
from datetime import timedelta
from concurrent.futures import ThreadPoolExecutor

In [ ]:
# Create the split adjuster
class SplitAdjuster:
    """
    Fetches and caches split histories from yfinance.
    For each (ticker, date) pair, computes how much yfinance has
    retroactively adjusted the price vs what it was quoted at on that day.

    The adjustment factor for a trade_date is:
        product of all split ratios for splits that occurred AFTER trade_date

    Example: NVDA on 2022-06-01
        Splits after that date: 2024-06-10 (10:1)
        Factor = 10
        yfinance adjusted close ~$68 -> unadjusted $680
        Polygon strikes from 2022 are ~$680 -> moneyness ~1.0 ✓
    """

    def __init__(self, cache_file=None):
        """
        Parameters
        ----------
        cache_file : str or None
            Path to a JSON file for persisting split histories across runs.
            If None, only in-memory caching is used.
        """
        self._memory_cache = {}   # {ticker: [(date_str, ratio), ...]}
        self._cache_file   = cache_file
        self._disk_cache   = {}

        if cache_file and os.path.exists(cache_file):
            try:
                with open(cache_file, "r") as f:
                    self._disk_cache = json.load(f)
                print(f"[SplitAdjuster] Loaded {len(self._disk_cache)} "
                      f"tickers from cache: {cache_file}")
            except Exception as e:
                print(f"[SplitAdjuster] Cache load error: {e}")

    def _fetch_splits(self, ticker):
        """
        Fetch split history for ticker from yfinance.
        Returns list of (date_str, ratio) sorted chronologically.
        """
        try:
            tk     = yf.Ticker(ticker)
            splits = tk.splits
            if splits.empty:
                return []
            result = []
            for dt, ratio in splits.items():
                # Normalise timezone-aware index to plain date string
                dt_clean = pd.to_datetime(dt).tz_localize(None).normalize()
                result.append((dt_clean.strftime("%Y-%m-%d"), float(ratio)))
            return sorted(result, key=lambda x: x[0])
        except Exception as e:
            print(f"[SplitAdjuster] yfinance split fetch error {ticker}: {e}")
            return []

    def get_splits(self, ticker):
        """
        Get split history for ticker, using cache when available.
        Returns list of (date_str, ratio).
        """
        # 1. In-memory cache
        if ticker in self._memory_cache:
            return self._memory_cache[ticker]

        # 2. Disk cache
        if ticker in self._disk_cache:
            splits = self._disk_cache[ticker]
            self._memory_cache[ticker] = splits
            return splits

        # 3. Fetch from yfinance
        splits = self._fetch_splits(ticker)
        self._memory_cache[ticker] = splits

        # Save to disk cache
        if self._cache_file:
            self._disk_cache[ticker] = splits
            try:
                with open(self._cache_file, "w") as f:
                    json.dump(self._disk_cache, f)
            except Exception:
                pass

        return splits

    def unadjust_factor(self, ticker, date_str):
        """
        Returns the factor to multiply a yfinance adjusted close by to
        recover the as-quoted price on date_str.

        Factor = product of all split ratios for splits that occurred
                 strictly AFTER date_str.

        If no splits occurred after date_str, returns 1.0 (no adjustment).
        """
        splits   = self.get_splits(ticker)
        trade_dt = pd.to_datetime(date_str)
        factor   = 1.0
        for split_date_str, ratio in splits:
            if pd.to_datetime(split_date_str) > trade_dt:
                factor *= ratio
        return factor

    def unadjust_close(self, ticker, date_str, adjusted_close):
        """
        Convert a yfinance adjusted close to as-quoted price on date_str.
        """
        if adjusted_close is None or np.isnan(adjusted_close):
            return adjusted_close
        factor = self.unadjust_factor(ticker, date_str)
        if factor == 1.0:
            return adjusted_close
        return round(adjusted_close * factor, 2)

    def get_unadjusted_closes(self, ticker, dates):
        """
        Bulk fetch adjusted closes from yfinance for all dates, then
        unadjust each one to match Polygon's historical strike prices.

        This is a drop-in replacement for _get_stock_closes_bulk().

        Parameters
        ----------
        ticker : str
        dates  : list of date strings (YYYY-MM-DD)

        Returns
        -------
        dict {date_str: unadjusted_close}
        """
        closes = {}
        if not dates:
            return closes

        all_dt   = [pd.to_datetime(d) for d in dates]
        min_date = (min(all_dt) - timedelta(days=7)).strftime("%Y-%m-%d")
        max_date = (max(all_dt) + timedelta(days=1)).strftime("%Y-%m-%d")

        try:
            df = yf.download(ticker, start=min_date, end=max_date,
                             progress=False, auto_adjust=True)
            if df.empty:
                return closes

            for date_str in dates:
                dt  = pd.to_datetime(date_str)
                sub = df[df.index <= dt]
                if not sub.empty:
                    val = sub["Close"].iloc[-1]
                    adj = round(
                        float(val.iloc[0] if hasattr(val, "iloc") else val), 2
                    )
                    closes[date_str] = self.unadjust_close(ticker, date_str, adj)
                else:
                    closes[date_str] = None

        except Exception as e:
            print(f"      [yfinance] ERROR {ticker}: {e}")

        return closes

    def diagnose(self, ticker, sample_date, polygon_client):
        """
        Diagnostic: print adjusted close, unadjusted close, and sample
        Polygon strikes for a given ticker and date. Use to verify the
        adjustment is correct before running the full pipeline.

        Usage:
            from polygon import RESTClient
            client   = RESTClient("YOUR_KEY")
            adjuster = SplitAdjuster()
            adjuster.diagnose("NVDA", "2022-06-01", client)
        """
        from datetime import timedelta as td
        closes = self.get_unadjusted_closes(ticker, [sample_date])
        unadj  = closes.get(sample_date)
        factor = self.unadjust_factor(ticker, sample_date)

        print(f"\n{'='*55}")
        print(f"Split Diagnostics: {ticker} on {sample_date}")
        print(f"Split history: {self.get_splits(ticker)}")
        print(f"Adjustment factor: {factor}")
        print(f"yfinance adjusted: "
              f"{round(unadj / factor, 2) if unadj else None}")
        print(f"After unadjust: {unadj}")

        # Sample Polygon strikes
        dt      = pd.to_datetime(sample_date)
        exp_min = (dt + td(days=10)).strftime("%Y-%m-%d")
        exp_max = (dt + td(days=60)).strftime("%Y-%m-%d")
        try:
            contracts = list(polygon_client.list_options_contracts(
                underlying_ticker=ticker,
                expiration_date_gte=exp_min,
                expiration_date_lte=exp_max,
                expired=True, limit=50,
            ))
            strikes = sorted(set(float(c.strike_price) for c in contracts))
            print(f"  Polygon strikes  : {strikes[:15]}")
            if unadj and strikes:
                nearest   = min(strikes, key=lambda s: abs(s - unadj))
                moneyness = nearest / unadj
                print(f"  Nearest strike   : {nearest}  "
                      f"(moneyness {moneyness:.4f})")
                status = "OK" if abs(moneyness - 1.0) < 0.05 else "WARNING"
                print(f"  Status           : {status}")
        except Exception as e:
            print(f"  Polygon error: {e}")

In [ ]:
# Configurations
from dotenv import load_dotenv
load_dotenv()

API_KEY    = os.get_env("polygon_key")
DATE_START = "2016-01-01"
DATE_END   = "2019-12-31"
YEARS      = list(range(2020, 2026))

OUTPUT_DIR              = "options_data_sp500_2020_2025"
CONSTITUENTS_CACHE_FILE = f"{OUTPUT_DIR}/sp500_constituents_by_year.json"
EARNINGS_FILE           = f"{OUTPUT_DIR}/sp500_earnings_history.csv"
OPTIONS_OUTPUT_FILE     = f"{OUTPUT_DIR}/sp500_options_pit.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Filters (from Gao, Xing & Zhang (2018) JFQA paper)
MIN_OPTION_PRICE = 0.125   # F1: min option price
MIN_STOCK_PRICE  = 5.0     # F2: min stock price
MIN_DTE          = 10      # F5: min days to expiry
MAX_DTE          = 60      # F5: max days to expiry
MIN_DELTA        = 0.375   # F6: min abs delta
MAX_DELTA        = 0.625   # F6: max abs delta
MIN_MONEYNESS    = 0.9     # F7: min moneyness
MAX_MONEYNESS    = 1.1     # F7: max moneyness
R                = 0.05
MIN_DTE_FOR_IV   = 3       # IV flagged unreliable below this (row still kept)

In [ ]:
# Split Adjuster
SPLIT_CACHE_FILE = f"{OUTPUT_DIR}/split_history_cache.json"
adjuster = SplitAdjuster(cache_file=SPLIT_CACHE_FILE)

In [ ]:
# Point in time S&P 500 constituents
SP500_HISTORY_CSV = "S&P 500 Historical Components & Changes(01-17-2026).csv"

def load_sp500_snapshot_csv(path):
    """
    Load the S&P 500 historical components CSV.
    Returns a DataFrame sorted by date with columns [date, tickers_set].
    """
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    # Pre-parse ticker sets for fast lookup
    df["ticker_list"] = df["tickers"].apply(
        lambda x: [t.strip().replace(".", "-") for t in str(x).split(",")
                   if t.strip()]
    )
    print(f"  Loaded {len(df):,} snapshots "
          f"({df['date'].min().date()} → {df['date'].max().date()})")
    return df


def get_constituents_on_date(snapshot_df, target_date):
    """
    Return the list of tickers in the S&P 500 on or before target_date.
    Finds the last snapshot row with date <= target_date.
    """
    target_dt = pd.to_datetime(target_date)
    eligible  = snapshot_df[snapshot_df["date"] <= target_dt]
    if eligible.empty:
        return []
    return eligible.iloc[-1]["ticker_list"]


def build_constituent_map(years, use_cache=True):
    """
    Build {year: {ticker: sector}} for all years using the local CSV.
    Direct snapshot lookup — no reconstruction, no approximation.
    Cached to disk after first run.
    """
    import datetime

    cache = {}
    if use_cache and os.path.exists(CONSTITUENTS_CACHE_FILE):
        with open(CONSTITUENTS_CACHE_FILE, "r") as f:
            cache = {int(k): v for k, v in json.load(f).items()}
        print(f"  Loaded constituent cache: {len(cache)} years")

    missing_years = [y for y in years if y not in cache or not cache[y]]
    if not missing_years:
        print("  All years cached.")
        return cache

    # Load CSV once for all missing years
    snapshot_df = load_sp500_snapshot_csv(SP500_HISTORY_CSV)

    for year in missing_years:
        if year >= datetime.date.today().year:
            target_date = datetime.date.today().strftime("%Y-%m-%d")
        else:
            target_date = f"{year}-12-31"

        tickers = get_constituents_on_date(snapshot_df, target_date)
        members = {t: "S&P500" for t in tickers}
        cache[year] = members
        print(f"  {year} (as of {target_date}): {len(members)} constituents")

    with open(CONSTITUENTS_CACHE_FILE, "w") as f:
        json.dump({str(k): v for k, v in cache.items()}, f, indent=2)
    print(f"  Constituent cache saved -> {CONSTITUENTS_CACHE_FILE}")

    return cache

In [ ]:
# Stock Closes - bulk fetch with automatic split unadjustment
def _get_stock_closes_bulk(ticker, dates):
    """
    Fetch adjusted closes then reverse-adjust using SplitAdjuster to match
    Polygon's historical (pre-split) strike price basis.
    Split history is fetched from yfinance once per ticker and cached.
    """
    return adjuster.get_unadjusted_closes(ticker, dates)

In [ ]:
# Polygon Option OHLC
def _option_ohlc(client, option_ticker, date_str):
    try:
        d    = pd.to_datetime(date_str).strftime("%Y-%m-%d")
        aggs = list(client.get_aggs(option_ticker, 1, "day", d, d))
        if aggs:
            a     = aggs[0]
            vwap  = float(a.vwap)  if (hasattr(a, "vwap")  and a.vwap)  else None
            close = float(a.close) if (hasattr(a, "close") and a.close) else None
            high  = float(a.high)  if (hasattr(a, "high")  and a.high)  else None
            low   = float(a.low)   if (hasattr(a, "low")   and a.low)   else None
            return vwap, close, high, low
    except Exception:
        pass
    return None, None, None, None

def _fetch_both_legs(client, call_ticker, put_ticker, date_str):
    with ThreadPoolExecutor(max_workers=2) as ex:
        fc = ex.submit(_option_ohlc, client, call_ticker, date_str)
        fp = ex.submit(_option_ohlc, client, put_ticker,  date_str)
        cv, cc, ch, cl = fc.result()
        pv, pc, ph, pl = fp.result()
    return cv, cc, ch, cl, pv, pc, ph, pl


def _get_mid(vwap, close):
    if vwap  is not None and vwap  > 0: return vwap,  "vwap"
    if close is not None and close > 0: return close, "close"
    return None, None

In [ ]:
# Black Scholes + Implied Vol Calculator
def _norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def _bs_price(S, K, T, r, sigma, opt):
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return float("nan")
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    if opt == "call":
        return S * _norm_cdf(d1) - K * math.exp(-r * T) * _norm_cdf(d2)
    return K * math.exp(-r * T) * _norm_cdf(-d2) - S * _norm_cdf(-d1)


def _implied_vol(price, S, K, T, r, opt,
                 lo=1e-6, hi=5.0, tol=1e-6, iters=120):
    if price is None or np.isnan(price) or price <= 0 or T <= 0:
        return float("nan")
    f_lo = _bs_price(S, K, T, r, lo, opt) - price
    f_hi = _bs_price(S, K, T, r, hi, opt) - price
    if np.isnan(f_lo) or np.isnan(f_hi) or f_lo * f_hi > 0:
        return float("nan")
    for _ in range(iters):
        mid   = 0.5 * (lo + hi)
        f_mid = _bs_price(S, K, T, r, mid, opt) - price
        if abs(f_mid) < tol: return mid
        if f_lo * f_mid <= 0: hi, f_hi = mid, f_mid
        else:                 lo, f_lo = mid, f_mid
    return 0.5 * (lo + hi)


def _bs_delta(S, K, T, r, sigma, opt):
    """Black-Scholes delta. Used for F6 filter: abs(delta) in [0.375, 0.625]."""
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return float("nan")
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    if opt == "call":
        return _norm_cdf(d1)
    return _norm_cdf(d1) - 1.0


def _get_trading_day(date_str, offset):
    current   = pd.to_datetime(date_str)
    direction = 1 if offset >= 0 else -1
    remaining = abs(offset)
    while remaining > 0:
        current += timedelta(days=direction)
        if current.weekday() < 5:
            remaining -= 1
    return current.strftime("%Y-%m-%d")

In [ ]:
# Entry Snapshot
def _get_entry_snapshot(client, ticker, entry_date, spot):
    """
    Find the ATM straddle on entry_date.
    Applies all Gao, Xing & Zhang (2018) filters:
      F1: option price >= MIN_OPTION_PRICE
      F2: stock price >= MIN_STOCK_PRICE
      F5: DTE between MIN_DTE and MAX_DTE
      F6: abs(delta) between MIN_DELTA and MAX_DELTA
      F7: moneyness between MIN_MONEYNESS and MAX_MONEYNESS
    """
    if spot is None or spot < MIN_STOCK_PRICE:
        return None

    trade_dt = pd.to_datetime(entry_date)
    exp_min  = (trade_dt + timedelta(days=MIN_DTE)).strftime("%Y-%m-%d")
    exp_max  = (trade_dt + timedelta(days=MAX_DTE)).strftime("%Y-%m-%d")

    try:
        contracts = list(client.list_options_contracts(
            underlying_ticker=ticker,
            expiration_date_gte=exp_min,
            expiration_date_lte=exp_max,
            expired=True, limit=1000,
        ))
    except Exception as e:
        print(f"    [entry contracts] ERROR {ticker} {entry_date}: {e}")
        return None
    if not contracts:
        return None

    df = pd.DataFrame([{
        "oticker": c.ticker,
        "strike":  float(c.strike_price),
        "expiry":  str(c.expiration_date),
        "type":    str(c.contract_type),
    } for c in contracts])

    # F7: moneyness filter
    df["moneyness"] = df["strike"] / spot
    df = df[df["moneyness"].between(MIN_MONEYNESS, MAX_MONEYNESS)]
    if df.empty:
        return None

    # Globally nearest strike first, then nearest expiry for that strike
    best_strike = min(df["strike"].unique(), key=lambda s: abs(s - spot))
    sdf = df[df["strike"] == best_strike]

    for expiry in sorted(sdf["expiry"].unique()):
        edf = sdf[sdf["expiry"] == expiry]
        dte = (pd.to_datetime(expiry) - trade_dt).days
        T   = max(dte / 365.25, 1e-6)

        cr = edf[edf["type"] == "call"]
        pr = edf[edf["type"] == "put"]
        if cr.empty or pr.empty:
            continue

        cv, cc, ch, cl, pv, pc, ph, pl = _fetch_both_legs(
            client, cr.iloc[0]["oticker"],
            pr.iloc[0]["oticker"], entry_date
        )
        call_mid, call_src = _get_mid(cv, cc)
        put_mid,  put_src  = _get_mid(pv, pc)

        # F9: price data available
        if call_mid is None or put_mid is None:
            continue

        # F1: minimum option price
        if call_mid < MIN_OPTION_PRICE or put_mid < MIN_OPTION_PRICE:
            continue

        call_iv = _implied_vol(call_mid, spot, best_strike, T, R, "call")
        put_iv  = _implied_vol(put_mid,  spot, best_strike, T, R, "put")

        # F6: abs(delta) filter
        call_sigma = call_iv if not np.isnan(call_iv) else 0.3
        put_sigma  = put_iv  if not np.isnan(put_iv)  else 0.3
        call_delta = _bs_delta(spot, best_strike, T, R, call_sigma, "call")
        put_delta  = _bs_delta(spot, best_strike, T, R, put_sigma,  "put")

        if np.isnan(call_delta) or np.isnan(put_delta):
            continue
        if not (MIN_DELTA <= abs(call_delta) <= MAX_DELTA):
            continue
        if not (MIN_DELTA <= abs(put_delta)  <= MAX_DELTA):
            continue

        atm_iv = float(np.nanmean([call_iv, put_iv]))

        return {
            "stock_close":  spot,
            "strike":       best_strike,
            "expiry":       expiry,
            "dte":          dte,
            "call_ticker":  cr.iloc[0]["oticker"],
            "call_vwap":    cv,  "call_close": cc,
            "call_mid":     call_mid,
            "call_high":    ch,  "call_low":   cl,
            "call_iv":      call_iv,
            "call_delta":   call_delta,
            "put_ticker":   pr.iloc[0]["oticker"],
            "put_vwap":     pv,  "put_close":  pc,
            "put_mid":      put_mid,
            "put_high":     ph,  "put_low":    pl,
            "put_iv":       put_iv,
            "put_delta":    put_delta,
            "straddle_mid": call_mid + put_mid,
            "price_source": f"{call_src}+{put_src}",
            "atm_iv":       atm_iv,
            "moneyness":    best_strike / spot,
        }

    return None

In [ ]:
# Exit Snapshot
def _get_exit_snapshot(client, ticker, exit_date,
                        pin_strike, pin_expiry, stock_exit):
    """
    Price the same contract (pin_strike, pin_expiry) on exit_date.
    Uses stock_exit for IV. No moneyness filter.
    """
    if stock_exit is None:
        print(f"    [exit] WARNING: no stock close for {ticker} "
              f"on {exit_date} — skipping")
        return None

    try:
        contracts = list(client.list_options_contracts(
            underlying_ticker=ticker,
            expiration_date_gte=pin_expiry,
            expiration_date_lte=pin_expiry,
            expired=True, limit=1000,
        ))
    except Exception as e:
        print(f"    [exit contracts] ERROR {ticker} {exit_date}: {e}")
        return None
    if not contracts:
        return None

    df = pd.DataFrame([{
        "oticker": c.ticker,
        "strike":  float(c.strike_price),
        "expiry":  str(c.expiration_date),
        "type":    str(c.contract_type),
    } for c in contracts])

    df = df[df["strike"] == pin_strike]
    if df.empty:
        return None

    cr = df[df["type"] == "call"]
    pr = df[df["type"] == "put"]
    if cr.empty or pr.empty:
        return None

    cv, cc, ch, cl, pv, pc, ph, pl = _fetch_both_legs(
        client, cr.iloc[0]["oticker"],
        pr.iloc[0]["oticker"], exit_date
    )
    call_mid, call_src = _get_mid(cv, cc)
    put_mid,  put_src  = _get_mid(pv, pc)

    if call_mid is None or put_mid is None:
        return None

    trade_dt    = pd.to_datetime(exit_date)
    dte_at_exit = (pd.to_datetime(pin_expiry) - trade_dt).days
    T           = max(dte_at_exit / 365.25, 1e-6)

    if dte_at_exit < MIN_DTE_FOR_IV:
        call_iv     = float("nan")
        put_iv      = float("nan")
        atm_iv      = float("nan")
        iv_reliable = False
    else:
        call_iv     = _implied_vol(call_mid, stock_exit, pin_strike, T, R, "call")
        put_iv      = _implied_vol(put_mid,  stock_exit, pin_strike, T, R, "put")
        atm_iv      = float(np.nanmean([call_iv, put_iv]))
        iv_reliable = True

    return {
        "stock_close":  stock_exit,
        "dte":          dte_at_exit,
        "iv_reliable":  iv_reliable,
        "call_vwap":    cv,  "call_close": cc,
        "call_mid":     call_mid,
        "call_high":    ch,  "call_low":   cl,
        "call_iv":      call_iv,
        "put_vwap":     pv,  "put_close":  pc,
        "put_mid":      put_mid,
        "put_high":     ph,  "put_low":    pl,
        "put_iv":       put_iv,
        "straddle_mid": call_mid + put_mid,
        "price_source": f"{call_src}+{put_src}",
        "atm_iv":       atm_iv,
    }



In [ ]:
# Run Process (Ticker, year) Job
ENTRY_OFFSETS = [-3, -2, -1, 0]
EXIT_OFFSETS  = [0, 1, 2, 3]
VALID_COMBOS  = [(e, x) for e in ENTRY_OFFSETS
                          for x in EXIT_OFFSETS if x >= e]


def process_ticker_year(ticker, year, sector, earn_dates,
                         job_num, total_jobs, already_done_events):
    """
    Run the full entry/exit grid for all earnings events of `ticker` in `year`.

    For each earnings event:
      1. Bulk fetch all stock closes needed across all offsets
      2. For each entry offset: get ATM entry snapshot (with all paper filters)
      3. For each valid (entry, exit) combo:
           - Same date → reuse entry snapshot
           - Different date → fetch exit snapshot pinned to entry's strike+expiry
      4. Cache exit snapshots per (exit_date, strike, expiry) to avoid
         redundant Polygon calls

    already_done_events: set of (ticker, earnings_date, entry_offset, exit_offset)
    """
    client = RESTClient(API_KEY)
    rows   = []

    year_start = f"{year}-01-01"
    year_end   = f"{year}-12-31"

    events = [
        d for d in earn_dates
        if year_start <= d <= year_end
    ]
    if not events:
        return rows

    # Bulk fetch all closes needed across all offsets in one yfinance call
    all_dates = set()
    for earn_date in events:
        for offset in ENTRY_OFFSETS + EXIT_OFFSETS:
            all_dates.add(_get_trading_day(earn_date, offset))
    closes = _get_stock_closes_bulk(ticker, list(all_dates))

    for earn_date in events:

        # ── Step A: entry snapshots for all entry offsets ─────────────────────
        entry_snaps = {}
        for e_off in ENTRY_OFFSETS:
            combos_needed = [
                (e_off, x_off) for x_off in EXIT_OFFSETS
                if x_off >= e_off
                and (ticker, earn_date, e_off, x_off) not in already_done_events
            ]
            if not combos_needed:
                continue

            e_date = _get_trading_day(earn_date, e_off)
            spot   = closes.get(e_date)
            snap   = _get_entry_snapshot(client, ticker, e_date, spot)
            if snap:
                entry_snaps[e_off] = (e_date, snap)

        if not entry_snaps:
            continue

        # ── Step B: exit snapshots, cached per (exit_date, strike, expiry) ────
        exit_cache = {}

        for e_off, (e_date, entry) in entry_snaps.items():
            for x_off in EXIT_OFFSETS:
                if x_off < e_off:
                    continue
                if (ticker, earn_date, e_off, x_off) in already_done_events:
                    continue

                x_date    = _get_trading_day(earn_date, x_off)
                spot_exit = closes.get(x_date)

                # Same date → reuse entry as exit (zero extra API call)
                if x_date == e_date:
                    exit_snap = {
                        "stock_close":  entry["stock_close"],
                        "dte":          entry["dte"],
                        "iv_reliable":  entry["dte"] >= MIN_DTE_FOR_IV,
                        "call_vwap":    entry["call_vwap"],
                        "call_close":   entry["call_close"],
                        "call_mid":     entry["call_mid"],
                        "call_high":    entry["call_high"],
                        "call_low":     entry["call_low"],
                        "call_iv":      entry["call_iv"],
                        "put_vwap":     entry["put_vwap"],
                        "put_close":    entry["put_close"],
                        "put_mid":      entry["put_mid"],
                        "put_high":     entry["put_high"],
                        "put_low":      entry["put_low"],
                        "put_iv":       entry["put_iv"],
                        "straddle_mid": entry["straddle_mid"],
                        "price_source": entry["price_source"],
                        "atm_iv":       entry["atm_iv"],
                    }
                else:
                    cache_key = (x_date, entry["strike"], entry["expiry"])
                    if cache_key not in exit_cache:
                        exit_cache[cache_key] = _get_exit_snapshot(
                            client, ticker, x_date,
                            pin_strike=entry["strike"],
                            pin_expiry=entry["expiry"],
                            stock_exit=spot_exit,
                        )
                    exit_snap = exit_cache[cache_key]

                if exit_snap is None:
                    continue

                stock_move = (
                    (exit_snap["stock_close"] - entry["stock_close"]) /
                     entry["stock_close"]
                    if exit_snap["stock_close"] and entry["stock_close"] else None
                )
                iv_change = (
                    exit_snap["atm_iv"] - entry["atm_iv"]
                    if (exit_snap["iv_reliable"] and
                        not np.isnan(exit_snap["atm_iv"]) and
                        not np.isnan(entry["atm_iv"]))
                    else None
                )
                pnl     = exit_snap["straddle_mid"] - entry["straddle_mid"]
                pnl_pct = pnl / entry["straddle_mid"] \
                          if entry["straddle_mid"] > 0 else None

                rows.append({
                    "index":                "SP500",
                    "ticker":               ticker,
                    "constituent_year":     year,
                    "sector":               sector,
                    "earnings_date":        earn_date,
                    "entry_offset":         e_off,
                    "exit_offset":          x_off,
                    "entry_date":           e_date,
                    "exit_date":            x_date,
                    "strike":               entry["strike"],
                    "expiry":               entry["expiry"],
                    "dte_at_entry":         entry["dte"],
                    "dte_at_exit":          exit_snap["dte"],
                    "exit_iv_reliable":     exit_snap["iv_reliable"],
                    "call_ticker":          entry["call_ticker"],
                    "put_ticker":           entry["put_ticker"],
                    "moneyness_at_entry":   entry["moneyness"],
                    "stock_entry":          entry["stock_close"],
                    "stock_exit":           exit_snap["stock_close"],
                    "stock_move":           stock_move,
                    "entry_call_vwap":      entry["call_vwap"],
                    "entry_call_close":     entry["call_close"],
                    "entry_call_mid":       entry["call_mid"],
                    "entry_call_high":      entry["call_high"],
                    "entry_call_low":       entry["call_low"],
                    "entry_call_iv":        entry["call_iv"],
                    "entry_call_delta":     entry.get("call_delta", float("nan")),
                    "entry_put_vwap":       entry["put_vwap"],
                    "entry_put_close":      entry["put_close"],
                    "entry_put_mid":        entry["put_mid"],
                    "entry_put_high":       entry["put_high"],
                    "entry_put_low":        entry["put_low"],
                    "entry_put_iv":         entry["put_iv"],
                    "entry_put_delta":      entry.get("put_delta", float("nan")),
                    "entry_straddle_mid":   entry["straddle_mid"],
                    "entry_price_source":   entry["price_source"],
                    "entry_atm_iv":         entry["atm_iv"],
                    "exit_call_vwap":       exit_snap["call_vwap"],
                    "exit_call_close":      exit_snap["call_close"],
                    "exit_call_mid":        exit_snap["call_mid"],
                    "exit_call_high":       exit_snap["call_high"],
                    "exit_call_low":        exit_snap["call_low"],
                    "exit_call_iv":         exit_snap["call_iv"],
                    "exit_put_vwap":        exit_snap["put_vwap"],
                    "exit_put_close":       exit_snap["put_close"],
                    "exit_put_mid":         exit_snap["put_mid"],
                    "exit_put_high":        exit_snap["put_high"],
                    "exit_put_low":         exit_snap["put_low"],
                    "exit_put_iv":          exit_snap["put_iv"],
                    "exit_straddle_mid":    exit_snap["straddle_mid"],
                    "exit_price_source":    exit_snap["price_source"],
                    "exit_atm_iv":          exit_snap["atm_iv"],
                    "iv_change":            iv_change,
                    "pnl_per_straddle":     pnl,
                    "pnl_pct":              pnl_pct,
                })

    if rows:
        print(f"  [{job_num}/{total_jobs}] {ticker} {year} "
              f"({sector}): {len(rows)} rows "
              f"({len(events)} events × up to {len(VALID_COMBOS)} combos)")
    return rows

In [ ]:
# Main
print("POINT-IN-TIME S&P 500 EARNINGS STRADDLE PIPELINE")
print(f"Index: S&P 500 (local CSV)  |  Years: {YEARS[0]}-{YEARS[-1]}")
print(f"Output: {OPTIONS_OUTPUT_FILE}")

# ── Step 1: Build point-in-time constituent map ───────────────────────────────
print(f"\n{'='*65}")
print("STEP 1: RECONSTRUCTING POINT-IN-TIME S&P 500 CONSTITUENTS")
print(f"{'='*65}")

constituent_map = build_constituent_map(YEARS)
# constituent_map = {year: {ticker: sector, ...}, ...}

# Summary
print(f"\nConstituent map summary:")
all_tickers_ever  = {}   # {ticker: sector} — use last known sector
for year, members in sorted(constituent_map.items()):
    print(f"  {year}: {len(members)} constituents")
    for ticker, sector in members.items():
        all_tickers_ever[ticker] = sector   # sector from most recent year

print(f"\nUnique tickers across all years: {len(all_tickers_ever)}")

# Step 2: Scrape earnings
print(f"\n{'='*65}")
print("STEP 2: SCRAPING EARNINGS DATES")
print(f"{'='*65}")

earnings_dict = scrape_earnings_for_tickers(sorted(all_tickers_ever.keys()))
# earnings_dict = {ticker: [date_str, ...]}

# Step 3: Build (ticker, year) job list
print(f"\n{'='*65}")
print("STEP 3: BUILDING JOB LIST")
print(f"{'='*65}")

jobs = []  # (ticker, year, sector, [earn_dates_in_that_year])
for year, members in sorted(constituent_map.items()):
    year_start = f"{year}-01-01"
    year_end   = f"{year}-12-31"
    for ticker, sector in members.items():
        earn_dates = [
            d for d in earnings_dict.get(ticker, [])
            if year_start <= d <= year_end
        ]
        if earn_dates:
            jobs.append((ticker, year, sector, earn_dates))

print(f"Total (ticker, year) jobs with earnings: {len(jobs)}")
print(f"Spanning {len(set(j[0] for j in jobs))} unique tickers "
      f"across {len(set(j[1] for j in jobs))} years")

# Sector breakdown
sector_counts = {}
for _, _, sector, _ in jobs:
    sector_counts[sector] = sector_counts.get(sector, 0) + 1
print(f"\nJobs by sector:")
for s, c in sorted(sector_counts.items(), key=lambda x: -x[1]):
    print(f"  {s}: {c}")

# Step 4: Options data collection 
print(f"\n{'='*65}")
print("STEP 4: COLLECTING OPTIONS DATA")
print(f"{'='*65}")
print(f"Filters (Gao, Xing & Zhang 2018):")
print(f"  F1: option price  >= ${MIN_OPTION_PRICE}")
print(f"  F2: stock price   >= ${MIN_STOCK_PRICE}")
print(f"  F5: DTE           {MIN_DTE}–{MAX_DTE} days")
print(f"  F6: abs(delta)    {MIN_DELTA}–{MAX_DELTA}")
print(f"  F7: moneyness     {MIN_MONEYNESS}–{MAX_MONEYNESS}")
print(f"Entry offsets: {ENTRY_OFFSETS}  |  Exit offsets: {EXIT_OFFSETS}")
print(f"Valid combos : {len(VALID_COMBOS)}")

# Per-combo resume — track (ticker, earnings_date, entry_offset, exit_offset)
already_done_events = set()
if os.path.exists(OPTIONS_OUTPUT_FILE):
    existing = pd.read_csv(OPTIONS_OUTPUT_FILE)
    already_done_events = set(
        zip(existing["ticker"], existing["earnings_date"],
            existing["entry_offset"], existing["exit_offset"])
    )
    print(f"\nResuming — {len(already_done_events):,} combo-rows already done "
          f"across {existing['ticker'].nunique()} tickers.")

total_jobs = len(jobs)
print(f"Jobs to process: {total_jobs}\n")

for job_num, (ticker, year, sector, earn_dates) in enumerate(jobs, 1):
    try:
        rows = process_ticker_year(
            ticker, year, sector, earn_dates,
            job_num, total_jobs, already_done_events
        )
    except Exception as e:
        print(f"  ⚠️  {ticker} {year} failed: {e}")
        rows = []

    if rows:
        batch_df     = pd.DataFrame(rows)
        write_header = not os.path.exists(OPTIONS_OUTPUT_FILE)
        batch_df.to_csv(
            OPTIONS_OUTPUT_FILE, mode="a",
            header=write_header, index=False
        )
        for r in rows:
            already_done_events.add((
                r["ticker"], r["earnings_date"],
                r["entry_offset"], r["exit_offset"]
            ))

    # Checkpoint every 200 jobs
    if job_num % 200 == 0 and os.path.exists(OPTIONS_OUTPUT_FILE):
        saved = pd.read_csv(OPTIONS_OUTPUT_FILE)
        pct   = job_num / total_jobs * 100
        print(f"\n── Checkpoint {job_num}/{total_jobs} ({pct:.1f}%) ──")
        print(f"   Rows saved:        {len(saved):,}")
        print(f"   Tickers with data: {saved['ticker'].nunique()}")
        print(f"   Years covered:     "
              f"{sorted(saved['constituent_year'].unique())}")

print(f"\n{'='*65}")
print("Pipeline complete!")
print(f"{'='*65}")

# Summary 
if os.path.exists(OPTIONS_OUTPUT_FILE):
    out = pd.read_csv(OPTIONS_OUTPUT_FILE)

    print(f"\n✓ {OPTIONS_OUTPUT_FILE}")
    print(f"  Total rows:       {len(out):,}")
    print(f"  Unique tickers:   {out['ticker'].nunique()}")
    print(f"  Years covered:    {sorted(out['constituent_year'].unique())}")
    print(f"  VWAP %:           "
          f"{(out['entry_price_source']=='vwap+vwap').mean()*100:.1f}%")
    print(f"  Exit IV reliable: "
          f"{out['exit_iv_reliable'].mean()*100:.1f}%")

    print(f"\nBy year:")
    print(out.groupby("constituent_year").agg(
        n          = ("ticker",        "count"),
        tickers    = ("ticker",        "nunique"),
        entry_iv   = ("entry_atm_iv",  lambda x: round(x.mean(), 3)),
        iv_chg     = ("iv_change",     lambda x: round(x.mean(), 4)),
        pnl_pct    = ("pnl_pct",       lambda x: round(x.mean(), 4)),
        stock_move = ("stock_move",    lambda x: round(x.mean(), 4)),
    ).to_string())

    print(f"\nBy sector:")
    print(out.groupby("sector").agg(
        n        = ("ticker",        "count"),
        tickers  = ("ticker",        "nunique"),
        entry_iv = ("entry_atm_iv",  lambda x: round(x.mean(), 3)),
        iv_chg   = ("iv_change",     lambda x: round(x.mean(), 4)),
        pnl_pct  = ("pnl_pct",       lambda x: round(x.mean(), 4)),
    ).sort_values("n", ascending=False).to_string())

    print(f"\nSample rows:")
    print(out[[
        "ticker", "constituent_year", "sector", "earnings_date",
        "entry_straddle_mid", "exit_straddle_mid",
        "pnl_per_straddle", "pnl_pct",
        "entry_atm_iv", "exit_atm_iv",
        "iv_change", "stock_move",
    ]].head(10).to_string(index=False))